# Real-time SEC filings: poll the API or stream new filings

Get new EDGAR filings live, as the SEC accepts them, by polling the filings endpoint or holding open a Server-Sent Events stream.

Read it with charts at https://www.quantisnow.com/sec-api/guides/real-time-sec-filings-api. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

New filings are searchable as soon as the SEC accepts them. There are two ways to pick them up: ask for the newest filings every so often, or keep a connection open and have each one pushed to you.

## Polling, on any plan

Sort by `filedAt:desc`, keep the ids you've seen, and ask again. Once a minute is 43,200 calls a month. The Hobby plan's 30,000 calls cover one query every two minutes, and Pro covers once a minute with room to spare. See [pricing](https://www.quantisnow.com/sec-api/pricing).

In [ ]:
import os, requests

BASE = "https://www.quantisnow.com/api/v1"
KEY = os.environ["QUANTISNOW_API_KEY"]

def search(endpoint, query, size=50, sort=None, start=0):
    body = {"query": query, "size": size, "from": start}
    if sort:
        body["sort"] = sort
    res = requests.post(f"{BASE}/{endpoint}", json=body,
                        headers={"Authorization": f"Bearer {KEY}"}, timeout=30)
    res.raise_for_status()
    return res.json()
import time

seen = set()
QUERY = 'formType:("8-K" OR "4") AND ticker:(AAPL OR NVDA OR TSLA)'
while True:
    today = time.strftime("%Y-%m-%d")
    page = search("filings", f"{QUERY} AND filedAt:>={today}", sort="filedAt:desc", size=50)
    for f in reversed(page["data"]):
        if f["id"] not in seen:
            seen.add(f["id"])
            print(f["filedAt"], f["formType"], f["ticker"], f["linkToFilingIndex"])
    time.sleep(60)

## Streaming, on the Business plan

`GET /v1/stream` is a Server-Sent Events stream of every filing we take in, as the same JSON the filings endpoint returns, filtered with the same query syntax. A connection lasts about four and a half minutes, then sends `reconnect`; the browser's `EventSource` reconnects by itself with `Last-Event-ID`, so nothing is lost in between. Each connection counts as one call.

```shell
curl -N "https://www.quantisnow.com/api/v1/stream?query=formType:8-K" \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY"
```

```javascript
// Node 22+: EventSource needs a header, so use fetch and read the stream
const res = await fetch("https://www.quantisnow.com/api/v1/stream?query=" + encodeURIComponent("formType:4 AND ticker:NVDA"), {
  headers: { Authorization: "Bearer " + process.env.QUANTISNOW_API_KEY },
});
const decoder = new TextDecoder();
let buf = "";
for await (const chunk of res.body) {
  buf += decoder.decode(chunk, { stream: true });
  let i;
  while ((i = buf.indexOf("\n\n")) >= 0) {
    const msg = buf.slice(0, i);
    buf = buf.slice(i + 2);
    const data = msg.split("\n").find((l) => l.startsWith("data: "));
    if (msg.includes("event: filing") && data) console.log(JSON.parse(data.slice(6)));
  }
}
```

In a browser, `new EventSource(url + "&token=" + key)` works too, because the server reads the key from `token`. Only do that with a key you're happy to expose to whoever uses the page.